# 09 - 每日增量管线

本 Notebook 完成以下任务：
1. 管线设计说明（早盘/午盘双时段）
2. 手动运行管线
3. cron 定时任务配置
4. 查看运行日志

---

## 管线设计

### 双时段调度

| 时段 | 触发时间 | 执行内容 | 目的 |
|------|---------|---------|------|
| **morning** | 11:35 | 日历 + 日线 + ETF + 指数 | 早盘行情快照 |
| **afternoon** | 15:05 | 全部 10 步 | 完整日终数据 + 校验 |

### morning 步骤（早盘后）
```
1. 交易日历同步        trade_calendar
2. 个股日线增量          stock_daily
3. ETF 日线增量          etf_daily
4. 指数日线增量          index_daily
```

### afternoon 步骤（午盘后）
```
1. 交易日历同步          trade_calendar
2. 股票/ETF 列表同步      stock_info, etf_info
3. 个股日线增量            stock_daily          ← 覆盖早盘半日数据
4. ETF 日线增量            etf_daily
5. 指数日线增量            index_daily
6. 每日估值增量            stock_fundamental
7. 融资融券增量            stock_margin
8. 股东增减持              stock_holder_trade
9. 股东户数                stock_holder_count
10. 数据质量校验           (不写表)
```

### 增量机制
所有行情类数据通过 `trade_calendar` 比对缺失交易日，只拉缺失部分。
午盘管线会用 Upsert 覆盖早盘的半日数据，保证最终是完整日线。

### 容错
每个步骤独立 try-except，单步失败不影响后续步骤。

## 1. 手动运行管线

In [1]:
from invest_model.pipeline.daily_pipeline import DailyPipeline

pipeline = DailyPipeline()

# session 可选: "morning" / "afternoon" / "full"
results = pipeline.run(session="full")

print("\n管线运行结果:")
for step, status in results.items():
    print(f"  {step}: {status}")

12:07:31 | INFO    | Tushare 客户端初始化完成
12:07:31 | INFO    | ============================================================
12:07:31 | INFO    | 每日管线启动 [full]: 2026-04-08 12:07:31
12:07:31 | INFO    | 执行步骤: ['calendar', 'stock_list', 'stock_daily', 'etf_daily', 'index_daily', 'daily_basic', 'margin', 'holder_trade', 'holder_count', 'validation']
12:07:31 | INFO    | ============================================================
12:07:31 | INFO    | --- [交易日历] 开始 ---
12:07:31 | INFO    | 采集交易日历: 20210101 ~ 20261231
12:07:31 | INFO    | 获取到 2191 条日历记录，其中交易日 1454 天
12:07:32 | INFO    | 写入 trade_calendar: 2191 条
12:07:32 | INFO    | --- [交易日历] 完成: 2191 ---
12:07:32 | INFO    | --- [股票列表] 开始 ---
12:07:32 | INFO    | 开始采集 A 股股票列表...
12:07:32 | INFO    | 获取到 5499 只 A 股
12:07:33 | INFO    | 写入 stock_info: 5499 条
12:07:33 | INFO    | 开始采集 ETF 列表...
12:07:33 | INFO    | 获取到 1941 只 ETF
12:07:34 | INFO    | 写入 etf_info: 1941 条
12:07:34 | INFO    | --- [股票列表] 完成: stocks=5499, etfs=1941 ---
12:07:34 | INF


管线运行结果:
  calendar: OK (2191)
  stock_list: OK (stocks=5499, etfs=1941)
  stock_daily: OK (5 只, 新增 0 条)
  etf_daily: OK (3 只, 新增 0 条)
  index_daily: OK (5 指数, 新增 0 条)
  daily_basic: OK (0)
  margin: OK (0)
  holder_trade: OK (42)
  holder_count: OK (92)
  validation: OK (checks=20, passed=15, warnings=5)


## 2. cron 定时任务配置

### 命令行运行
```bash
cd /home/admin/Code/invest-journey/invest-model

# 完整管线
python3 -m invest_model.pipeline.daily_pipeline full

# 只跑早盘
python3 -m invest_model.pipeline.daily_pipeline morning

# 只跑午盘
python3 -m invest_model.pipeline.daily_pipeline afternoon
```

### crontab 配置
周一到周五，早盘收盘后 11:35 和午盘收盘后 15:05 各执行一次：

```bash
# 编辑 crontab
crontab -e

# 添加以下两行
35 11 * * 1-5 cd /home/admin/Code/invest-journey/invest-model && /usr/bin/python3 -m invest_model.pipeline.daily_pipeline morning   >> logs/cron.log 2>&1
 5 15 * * 1-5 cd /home/admin/Code/invest-journey/invest-model && /usr/bin/python3 -m invest_model.pipeline.daily_pipeline afternoon >> logs/cron.log 2>&1
```

### 验证 crontab
```bash
crontab -l
```

## 3. 查看运行日志

In [2]:
from pathlib import Path
from invest_model.config import get_project_root

log_dir = get_project_root() / "logs"
log_files = sorted(log_dir.glob("invest-model-*.log"), reverse=True)

if log_files:
    latest = log_files[0]
    print(f"最新日志: {latest.name}")
    lines = latest.read_text().strip().split("\n")
    print(f"共 {len(lines)} 行，最后 20 行:\n")
    for line in lines[-20:]:
        print(line)
else:
    print("暂无日志文件")

最新日志: invest-model-20260408.log
共 13332 行，最后 20 行:

2026-04-08 12:10:13 | DEBUG   | base:upsert:87 | Upsert stock_holder_count: 15 rows
2026-04-08 12:10:13 | DEBUG   | base:upsert:87 | Upsert stock_holder_count: 13 rows
2026-04-08 12:10:13 | DEBUG   | base:upsert:87 | Upsert stock_holder_count: 18 rows
2026-04-08 12:10:13 | INFO    | event_collector:collect_holder_count:77 | 股东户数采集完成: 5 只, 共 92 条
2026-04-08 12:10:13 | INFO    | daily_pipeline:_run_step:126 | --- [股东户数] 完成: 92 ---
2026-04-08 12:10:13 | INFO    | daily_pipeline:_run_step:124 | --- [数据校验] 开始 ---
2026-04-08 12:10:14 | INFO    | daily_pipeline:_run_step:126 | --- [数据校验] 完成: checks=20, passed=15, warnings=5 ---
2026-04-08 12:10:14 | INFO    | daily_pipeline:run:113 | ============================================================
2026-04-08 12:10:14 | INFO    | daily_pipeline:run:114 | 每日管线完成 [full]，耗时 162.8s
2026-04-08 12:10:14 | INFO    | daily_pipeline:run:116 |   calendar: OK (2191)
2026-04-08 12:10:14 | INFO    | daily_pip

## 完成

每日管线已配置完毕。数据层全部 10 个 Notebook 已完成：

| 序号 | Notebook | 状态 |
|------|---------|------|
| 00 | 环境配置 | Done |
| 01 | 数据库初始化 | Done |
| 02 | 股票池管理 | Done |
| 03 | 日线行情 | Done |
| 04 | 财务指标 | Done |
| 05 | ETF 数据 | Done |
| 06 | 事件数据 | Done |
| 07 | 市场数据 | Done |
| 08 | 数据校验 | Done |
| 09 | 每日管线 | Done |